# 1. ライブラリインポート・定数などの設定

In [ ]:
import os
import sys
import json
import sqlite3
import pandas as pd
import pandas_ta as ta
import numpy as np
import matplotlib.pyplot as plt

# データパスの設定
DB_PATH = "../data/stock_data.db"
TICKER_DICT_PATH = "../data/ticker_dictionary.json"

# ノートブックでのグラフ表示用設定
%matplotlib inline
plt.rcParams['font.family'] = 'sans-serif'

print("Libraries imported successfully.")


# 2. ターゲット設定（対象銘柄、期間、および市場指数）

In [ ]:
# 検出したい特定銘柄群（銘柄コードのリスト）
TARGET_CODES = ['6976', '6762', '6981'] # 太陽誘電, TDK, 村田製作所

# 特定しておきたかった期間
START_DATE = '2026-04-01'
END_DATE = '2026-05-25'

# 相対強度を測るための市場指数（例として'0000'をダミーのTOPIXとして扱う）
INDEX_CODE = '0000'

print(f"ターゲット銘柄: {TARGET_CODES}")
print(f"対象期間: {START_DATE} 〜 {END_DATE}")


# 3. データの読み込みと共通テーマ（タグ）の特定

In [ ]:
with open(TICKER_DICT_PATH, 'r', encoding='utf-8') as f:
    ticker_dict = json.load(f)

code_to_name = {}
code_to_tags = {}

for item in ticker_dict:
    code = item['code']
    code_to_name[code] = item['name']
    code_to_tags[code] = item.get('tags', [])

target_names = []
common_tags = None

for code in TARGET_CODES:
    name = code_to_name.get(code, "Unknown")
    target_names.append(name)
    tags = set(code_to_tags.get(code, []))
    
    if common_tags is None:
        common_tags = tags
    else:
        common_tags = common_tags.intersection(tags)

print("【ターゲット銘柄の共通タグ（積集合）】")
if common_tags:
    for tag in common_tags:
        print(f"★ {tag}")
else:
    print("共通タグは見つかりませんでした。")


# 4. 株価データの取得とマルチ指標・特徴量計算

In [ ]:
def calculate_advanced_metrics(df_group, df_index):
    # ソートと欠損値処理
    df = df_group.sort_values('date_dt').copy()
    
    # NaNや0埋め（流動性がない日など）の簡易補完（前方補完）
    df = df.ffill().fillna(0)
    
    # ---------------------------------------------------------
    # 1. 資金流入・蓄積系 (Volume/Money Flow)
    # ---------------------------------------------------------
    # 出来高モメンタム
    df['vol_5d_avg'] = df['volume'].rolling(window=5, min_periods=1).mean()
    df['vol_25d_avg'] = df['volume'].rolling(window=25, min_periods=1).mean()
    df['vol_ratio'] = np.where(df['vol_25d_avg'] > 0, df['vol_5d_avg'] / df['vol_25d_avg'], 0)
    
    # OBV (On-Balance Volume)
    if len(df) > 1:
        df['obv'] = ta.obv(df['close'], df['volume'])
        # OBVの20日傾き（モメンタム）
        df['obv_slope_20d'] = df['obv'].diff(20)
        # OBVの60日新高値更新フラグ
        df['obv_high_60d'] = df['obv'].rolling(window=60, min_periods=1).max()
        df['obv_new_high'] = (df['obv'] >= df['obv_high_60d']).astype(int)
    else:
        df['obv'] = 0
        df['obv_slope_20d'] = 0
        df['obv_new_high'] = 0
        
    # CMF (Chaikin Money Flow, 20日)
    if len(df) >= 20:
        df['cmf_20d'] = ta.cmf(df['high'], df['low'], df['close'], df['volume'], length=20)
    else:
        df['cmf_20d'] = 0

    # ---------------------------------------------------------
    # 2. 相対強度・セクターアウトパフォーム系 (Relative Strength)
    # ---------------------------------------------------------
    # インデックスとのマージによる相対強度計算
    if not df_index.empty:
        df = pd.merge(df, df_index[['date_dt', 'close']], on='date_dt', how='left', suffixes=('', '_idx'))
        df['close_idx'] = df['close_idx'].ffill()
        df['rs_index'] = np.where(df['close_idx'] > 0, df['close'] / df['close_idx'], np.nan)
        # 相対強度の過去60日高値更新チェック
        df['rs_high_60d'] = df['rs_index'].rolling(window=60, min_periods=1).max()
        df['rs_breakout'] = (df['rs_index'] >= df['rs_high_60d']).astype(int)
    else:
        df['rs_index'] = np.nan
        df['rs_breakout'] = 0

    # ---------------------------------------------------------
    # 3. ボラティリティ・値幅拡大系 (Volatility/ATR)
    # ---------------------------------------------------------
    if len(df) >= 20:
        # ATRの計算
        df['atr_5d'] = ta.atr(df['high'], df['low'], df['close'], length=5)
        df['atr_20d'] = ta.atr(df['high'], df['low'], df['close'], length=20)
        df['atr_ratio'] = np.where(df['atr_20d'] > 0, df['atr_5d'] / df['atr_20d'], 0)
    else:
        df['atr_ratio'] = 0
        
    # 高安値幅の拡大比率: (High - Low) / 20日平均高安値幅
    df['high_low_spread'] = df['high'] - df['low']
    df['avg_spread_20d'] = df['high_low_spread'].rolling(window=20, min_periods=1).mean()
    df['spread_expansion'] = np.where(df['avg_spread_20d'] > 0, df['high_low_spread'] / df['avg_spread_20d'], 0)
    
    # ---------------------------------------------------------
    # 4. 需給・回転率系 (Turnover Ratio)
    # ---------------------------------------------------------
    # 発行済株式数として、擬似的に 時価総額 / 終値 を使用（データがある場合）
    df['shares_out'] = np.where(df['close'] > 0, df['market_cap_total'] * 1000000 / df['close'], np.nan)
    df['turnover_ratio'] = np.where((df['shares_out'] > 0) & df['shares_out'].notna(), df['volume'] / df['shares_out'], 0)
    # 過去20日平均回転率からの乖離率
    df['avg_turnover_20d'] = df['turnover_ratio'].rolling(window=20, min_periods=1).mean()
    df['turnover_spike'] = np.where(df['avg_turnover_20d'] > 0, df['turnover_ratio'] / df['avg_turnover_20d'], 0)

    # 既存の価格位置 (price_vs_high)
    df['high_250d'] = df['close'].rolling(window=250, min_periods=1).max()
    df['price_vs_high'] = np.where(df['high_250d'] > 0, df['close'] / df['high_250d'], 0)

    return df

def fetch_and_calculate_all(db_path, target_codes, index_code):
    conn = sqlite3.connect(db_path)
    
    # 対象銘柄とインデックスのコードをまとめる
    all_codes = list(set(target_codes + [index_code]))
    placeholders = ','.join('?' for _ in all_codes)
    
    query = f"""
    SELECT date, code, open, high, low, close, volume, market_cap_total
    FROM daily_prices 
    WHERE code IN ({placeholders})
    ORDER BY code, date
    """
    
    df_raw = pd.read_sql_query(query, conn, params=all_codes)
    conn.close()
    
    df_raw['code'] = df_raw['code'].astype(str)
    df_raw['date_dt'] = pd.to_datetime(df_raw['date'], format='%Y%m%d')
    for col in ['open', 'high', 'low', 'close', 'volume', 'market_cap_total']:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
        
    # インデックスデータの分離
    df_index = df_raw[df_raw['code'] == index_code].sort_values('date_dt').copy()
    
    # ターゲット銘柄の処理
    processed_dfs = []
    for code in target_codes:
        df_group = df_raw[df_raw['code'] == code].copy()
        if df_group.empty:
            continue
        df_processed = calculate_advanced_metrics(df_group, df_index)
        processed_dfs.append(df_processed)
        
    if processed_dfs:
        df_all_processed = pd.concat(processed_dfs, ignore_index=True)
        return df_all_processed
    else:
        return pd.DataFrame()

df_all = fetch_and_calculate_all(DB_PATH, TARGET_CODES, INDEX_CODE)

start_dt = pd.to_datetime(START_DATE)
end_dt = pd.to_datetime(END_DATE)

df_target = df_all[(df_all['date_dt'] >= start_dt) & (df_all['date_dt'] <= end_dt)].copy()
print(f"データ取得・指標計算完了: 対象期間 {len(df_target)} レコード")


# 5. 新指標の推移可視化 (OBV / CMF / ATR拡大率)

In [ ]:
for code in TARGET_CODES:
    df_plot = df_target[df_target['code'] == code].copy()
    if df_plot.empty:
        continue
        
    name = code_to_name.get(code, "Unknown")
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    fig.suptitle(f"{name} ({code}) - Advanced Momentum Indicators", fontsize=16)
    
    # 1. OBV
    axes[0].plot(df_plot['date_dt'], df_plot['obv'], color='purple', label='OBV')
    axes[0].set_ylabel('OBV')
    axes[0].legend(loc='upper left')
    axes[0].grid(True, alpha=0.3)
    
    # 2. CMF (Chaikin Money Flow)
    axes[1].plot(df_plot['date_dt'], df_plot['cmf_20d'], color='orange', label='CMF (20d)')
    axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
    axes[1].set_ylabel('CMF')
    axes[1].legend(loc='upper left')
    axes[1].grid(True, alpha=0.3)
    
    # 3. ボラティリティ (ATR Ratio)
    axes[2].plot(df_plot['date_dt'], df_plot['atr_ratio'], color='green', label='ATR Ratio (5d/20d)')
    axes[2].axhline(1.0, color='black', linestyle='--', alpha=0.5)
    axes[2].set_ylabel('ATR Ratio')
    axes[2].set_xlabel('Date')
    axes[2].legend(loc='upper left')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# 6. 検知に必要な拡張パラメータ（閾値）のボトムアップ逆算とCSV出力

In [ ]:
print(f"【{START_DATE} 〜 {END_DATE} の期間において各銘柄が記録した最大モメンタム】\n")

for code in TARGET_CODES:
    df_code = df_target[df_target['code'] == code]
    if df_code.empty:
        continue
    
    name = code_to_name.get(code, "Unknown")
    
    max_vol_ratio = df_code['vol_ratio'].max()
    max_cmf = df_code['cmf_20d'].max()
    max_atr_ratio = df_code['atr_ratio'].max()
    max_turnover_spike = df_code['turnover_spike'].max()
    max_price_vs_high = df_code['price_vs_high'].max()
    
    print(f"◆ {name} ({code})")
    print(f"   - 最大 出来高倍率        : {max_vol_ratio:.2f} 倍")
    print(f"   - 最大 高値圏位置        : {max_price_vs_high:.3f}")
    print(f"   - 最大 CMF (20d)         : {max_cmf:.3f} (資金流入)")
    print(f"   - 最大 ATR拡大率         : {max_atr_ratio:.2f} 倍 (ボラティリティ急増)")
    print(f"   - 最大 回転率スパイク    : {max_turnover_spike:.2f} 倍 (需給の急変)\n")

# 統合データをCSVに出力
OUTPUT_CSV = 'forensic_analysis_results.csv'
cols_to_export = [
    'date_dt', 'code', 'close', 'volume', 'vol_ratio', 'price_vs_high', 
    'obv', 'obv_slope_20d', 'obv_new_high', 'cmf_20d', 'rs_index', 'rs_breakout', 
    'atr_ratio', 'spread_expansion', 'turnover_spike'
]
df_export = df_target[cols_to_export].copy()
df_export.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f"✅ 全指標を含む統合データを '{OUTPUT_CSV}' に出力しました。")
